# Review 2: full-data Task 2 targeted replay-label poisoning and consistency filtering

Three fixed conditions only: clean replay, targeted-20% old-attack-to-Benign replay labels, and the **same** corrupted replay after a frozen-teacher consistency gate. This is a class-incremental Task 2 comparison, not Task 1 retraining or a production defense. Attach the private prepared folder and the entire completed Task 1 run folder to Kaggle, enable Internet and T4 GPU, then set both selectors below. The combined flow collection has unverified per-row source provenance. Do not use test results to change these settings.


## 1. Pin published source and verify the Kaggle runtime

The pinned source includes bounded Task 2 replay sampling, durable epoch-boundary training, label flipping, consistency scoring, and count-based evaluation. Preserve Kaggle's CUDA-enabled PyTorch installation. Focused synthetic tests must discover nonzero tests and pass without skips.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, platform, re, random
from dataclasses import asdict, replace
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '857dfbbcca2cfd6b33041ded51fbbae39b7a43f2'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-full-task2')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.data import BROAD_CLASS_ORDER, FittedPreprocessor
from src.data.large_data import large_data_paths, verify_full_partitions
from src.models import TabularTransformerClassifier
from src.training.disk_backed import DiskBackedFlowDataset
from src.training.task2_replay import Task2ReplayView
from src.training.full_run import train_full_disk_backed
from src.poisoning import apply_label_flip
from src.mitigation import label_inconsistency_scores, calibrate_label_consistency_scores, apply_label_consistency_gate
from src.evaluation import replay_gate_metrics
from src.evaluation.streaming import evaluate_disk_backed, summarize_task2_counts
from src.evaluation.plots import save_result_plots
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable a Kaggle T4 GPU and restart')
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
environment = {'python': platform.python_version(), 'numpy': version('numpy'), 'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'cuda_device_count': torch.cuda.device_count()}
print('Verified source:', checked_out_sha, 'environment:', environment)
for pattern in ('test_task2_replay.py', 'test_full_continual_metrics.py', 'test_label_consistency.py', 'test_full_disk_training.py'):
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern, '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
    print(result.stdout, end=''); print(result.stderr, end='')
    summary = result.stdout + result.stderr
    count = re.search(r'Ran (\d+) tests? in', summary)
    if count is None or int(count.group(1)) == 0 or re.search(r'OK \(skipped=\d+\)', summary):
        raise RuntimeError(f'Focused tests absent or skipped: {pattern}')


## 2. Select and validate two read-only inputs

`PREPARED_DIR` is the prior full-row preparation output; `TASK1_RUN_DIR` is the **completed five-class Task 1 run** containing `best.pt`, replay data and manifests. Both may be current `/kaggle/working` folders or attached `/kaggle/input` folders. Stop on missing/ambiguous selectors. We verify source/input identity, every split, class IDs, the continual preprocessing hash and scope, the checkpoint hash, and replay shape, labels, original IDs and training-only provenance. The static eight-class checkpoint/preprocessor are never loaded.


In [ ]:
PREPARED_DIR = None  # edit: '/kaggle/input/.../review2_large_prepare_...' or '/kaggle/working/...'
TASK1_RUN_DIR = None  # edit: '/kaggle/input/.../full_task1_...' or '/kaggle/working/...'
EXPECTED_PREP_SOURCE_SHA = '5c0d519596bc6c4b99948126443635c2f7eb15f1'
EXPECTED_TASK1_SOURCE_SHA = 'e6ebfcf7a9beec4d2b8fd5245e7c62314b9fadf9'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
OLD_IDS, NEW_IDS, ALL_IDS = tuple(range(5)), (5, 6, 7), tuple(range(8))
def choose_folder(explicit, prefix, marker):
    if explicit is None:
        candidates = [p.parent for base in (Path('/kaggle/working'), Path('/kaggle/input')) for p in base.rglob(marker) if prefix in p.parent.name]
        print(prefix, 'candidates:', [str(p) for p in candidates])
        if len(candidates) != 1:
            raise RuntimeError(f'Found {len(candidates)} {prefix} folders; set its selector explicitly')
        folder = candidates[0]
    else:
        folder = Path(explicit)
    if not folder.is_dir() or not (folder.is_relative_to(Path('/kaggle/working')) or folder.is_relative_to(Path('/kaggle/input'))):
        raise ValueError(f'{prefix} must be a Kaggle working/input folder')
    return folder
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
prepared_dir = choose_folder(PREPARED_DIR, 'review2_large_prepare_', 'manifest.json')
task1_dir = choose_folder(TASK1_RUN_DIR, 'full_task1_', 'final_manifest.json')
assert task1_dir != prepared_dir
prep_manifest = json.loads((prepared_dir / 'manifest.json').read_text())
split_manifest = json.loads((prepared_dir / 'split_manifest.json').read_text())
features_manifest = json.loads((prepared_dir / 'features_manifest.json').read_text())
continual_record = json.loads((prepared_dir / 'preprocessing_continual.json').read_text())
task1_final = json.loads((task1_dir / 'final_manifest.json').read_text())
task1_config = json.loads((task1_dir / 'run_config.json').read_text())
task1_training = json.loads((task1_dir / 'training_manifest.json').read_text())
task1_reload = json.loads((task1_dir / 'reload_verification.json').read_text())
task1_reference = json.loads((task1_dir / 'task1_test_metrics.json').read_text())
replay_manifest = json.loads((task1_dir / 'replay_manifest.json').read_text())
assert prep_manifest['completed_preparation_and_benchmark_only'] is True
assert prep_manifest['source_sha'] == EXPECTED_PREP_SOURCE_SHA and prep_manifest['input_sha256'] == EXPECTED_INPUT_SHA256
assert prep_manifest['input_rows'] == split_manifest['row_count'] == features_manifest['row_count'] == 9_167_581
assert split_manifest['seed'] == 42 and split_manifest['class_order'] == list(BROAD_CLASS_ORDER) and split_manifest['split_codes'] == {'train': 0, 'validation': 1, 'test': 2}
assert features_manifest['dtype'] == 'float32' and features_manifest['targets_excluded'] == ['Label', 'ClassLabel'] and len(features_manifest['feature_names']) == 54
assert continual_record['scope'] == 'continual' and continual_record['fit_rows'] == 6_347_232 and continual_record['feature_names'] == features_manifest['feature_names']
assert continual_record['reservoir_seed'] == 42 and continual_record['reservoir_size_used'] == 200_000
paths = large_data_paths(prepared_dir)
actual_split = verify_full_partitions(paths, 9_167_581)
assert actual_split['counts'] == split_manifest['counts']
assert sum(split_manifest['counts'][BROAD_CLASS_ORDER[i]]['train'] for i in NEW_IDS) == 70_076
assert sum(split_manifest['counts'][BROAD_CLASS_ORDER[i]]['train'] for i in OLD_IDS) == 6_347_232
assert task1_final['status'] == 'completed full-data Task 1 foundation only' and task1_final['task2_trained'] is False
assert task1_final['source_sha'] == task1_config['source_sha'] == EXPECTED_TASK1_SOURCE_SHA
assert task1_final['class_ids'] == task1_config['class_ids'] == task1_training['class_ids'] == list(OLD_IDS)
assert task1_final['completed_epoch'] == 12 and task1_final['best_epoch'] == 9
assert task1_config['model']['num_classes'] == 5 and task1_config['model']['epochs'] == 12 and task1_config['model']['seed'] == 42
assert task1_config['preprocessing_scope'] == 'continual Task 1 training only' and task1_final['training_rows'] == 6_347_232
assert task1_final['data_identity'] == task1_config['data_identity'] == task1_training['data_identity'] == task1_reference['data_identity']
task1_identity = task1_final['data_identity']
assert task1_identity['input_sha256'] == EXPECTED_INPUT_SHA256 and task1_identity['prep_source_sha'] == EXPECTED_PREP_SOURCE_SHA
assert task1_identity['preprocessing_scope'] == 'Task 1 training global IDs 0-4 only' and task1_identity['task1_class_ids'] == list(OLD_IDS)
for path, key in ((paths.raw_features, 'raw_features_sha256'), (paths.labels, 'labels_sha256'), (paths.splits, 'splits_sha256'), (paths.row_ids, 'row_ids_sha256'), (prepared_dir / 'split_manifest.json', 'split_manifest_sha256'), (prepared_dir / 'preprocessing_continual.json', 'continual_preprocessing_sha256')):
    assert file_sha256(path) == task1_identity[key], f'Prepared identity mismatch: {key}'
assert file_sha256(task1_dir / 'preprocessing_continual.json') == task1_identity['continual_preprocessing_sha256']
best_path = task1_dir / 'best.pt'
task1_best_sha = file_sha256(best_path)
assert task1_best_sha == task1_final['best_checkpoint_sha256'] == task1_reload['best_checkpoint_sha256'] == task1_reference['best_checkpoint_sha256']
checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
assert checkpoint['class_ids'] == list(OLD_IDS) and checkpoint['data_identity'] == task1_identity and checkpoint['best_epoch'] == task1_final['best_epoch']
teacher = TabularTransformerClassifier.load(best_path, device='cuda')
assert teacher.class_ids == OLD_IDS
teacher.network.eval()
for parameter in teacher.network.parameters(): parameter.requires_grad_(False)
preprocessor = FittedPreprocessor(tuple(continual_record['feature_names']), np.asarray(continual_record['imputation_values'], dtype=np.float64), np.asarray(continual_record['means'], dtype=np.float64), np.asarray(continual_record['scales'], dtype=np.float64))
assert all(np.isfinite(a).all() for a in (preprocessor.imputation_values, preprocessor.means, preprocessor.scales)) and np.all(preprocessor.scales > 0)
with np.load(task1_dir / 'replay_buffer.npz', allow_pickle=False) as saved:
    replay_x = saved['features'].copy(); replay_y = saved['labels'].copy(); replay_ids = saved['original_row_ids'].copy()
assert replay_x.shape == (500, 54) and replay_x.dtype == np.float32 and np.isfinite(replay_x).all()
assert replay_y.shape == replay_ids.shape == (500,) and replay_y.dtype == replay_ids.dtype == np.int64
assert len(np.unique(replay_ids)) == 500 and np.array_equal(replay_ids, np.asarray(replay_manifest['original_row_ids'], dtype=np.int64))
assert replay_manifest['seed'] == 42 and replay_manifest['class_ids'] == list(OLD_IDS) and replay_manifest['rows_per_class'] == 100
assert replay_manifest['preprocessing_sha256'] == task1_identity['continual_preprocessing_sha256'] and replay_manifest['source_partition'] == 'original training only'
assert all(np.count_nonzero(replay_y == i) == replay_manifest['counts'][str(i)] == 100 for i in OLD_IDS)
labels_map = np.load(paths.labels, mmap_mode='r', allow_pickle=False)
splits_map = np.load(paths.splits, mmap_mode='r', allow_pickle=False)
raw_map = np.load(paths.raw_features, mmap_mode='r', allow_pickle=False)
row_map = np.load(paths.row_ids, mmap_mode='r', allow_pickle=False)
assert np.all(splits_map[replay_ids] == 0) and np.array_equal(row_map[replay_ids], replay_ids) and np.array_equal(labels_map[replay_ids], replay_y)
assert np.array_equal(preprocessor.transform(raw_map[replay_ids]), replay_x), 'Replay features differ from frozen Task 1 transformation'
assert task1_reference['metrics']['class_ids'] == list(OLD_IDS) and task1_reference['metrics']['total_rows'] == task1_final['old_test_rows']
print('Verified prepared input, Task 1 checkpoint hash/head/state, and replay training provenance; no input was modified')


## 3. Create or resume a separate auditable Task 2 output folder

`RESUME_RUN_DIR` may point to a prior `/kaggle/working` run or to a saved Kaggle input version. Attached inputs are copied to a new writable `/kaggle/working` folder; prepared and Task 1 inputs stay read-only. Each condition has its own latest/best checkpoints and history. If a session stops mid-epoch, only that epoch repeats. Strict data/config/runtime compatibility is checked by the durable trainer.


In [ ]:
RESUME_RUN_DIR = None  # optional previous full_task2_targeted_... folder under /kaggle/working or /kaggle/input
MAX_EPOCHS_THIS_CALL = None  # optional positive integer to pause each active arm at a completed epoch
if RESUME_RUN_DIR is None:
    output_dir = Path('/kaggle/working') / ('full_task2_targeted_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    output_dir.mkdir(exist_ok=False)
else:
    previous_output = Path(RESUME_RUN_DIR)
    if not previous_output.is_dir() or not (previous_output.is_relative_to(Path('/kaggle/working')) or previous_output.is_relative_to(Path('/kaggle/input'))):
        raise ValueError('RESUME_RUN_DIR must be a prior Kaggle output folder')
    if previous_output.is_relative_to(Path('/kaggle/working')):
        output_dir = previous_output
    else:
        output_dir = Path('/kaggle/working') / ('full_task2_targeted_resume_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        shutil.copytree(previous_output, output_dir)
assert output_dir != prepared_dir and output_dir != task1_dir and not output_dir.is_relative_to(prepared_dir) and not output_dir.is_relative_to(task1_dir)
if shutil.disk_usage('/kaggle/working').free < 1_000_000_000:
    raise RuntimeError('Less than 1 GB free for three arms and durable checkpoints')
conditions = ('clean_replay', 'targeted20_unfiltered', 'targeted20_filtered')
protocol = {'source_sha': checked_out_sha, 'prepared_input_sha256': EXPECTED_INPUT_SHA256, 'task1_source_sha': EXPECTED_TASK1_SOURCE_SHA, 'task1_best_sha256': task1_best_sha, 'task1_data_identity': task1_identity, 'preprocessing_sha256': task1_identity['continual_preprocessing_sha256'], 'condition_names': list(conditions), 'attack': {'mode': 'targeted', 'rate': 0.20, 'seed': 42, 'source_class_ids': [1, 2, 3, 4], 'target_class_id': 0, 'eligible_denominator': 400, 'total_replay_rows': 500}, 'gate': {'teacher': 'frozen clean Task 1 best', 'calibration': '95th percentile of clean Task 1 validation inconsistency', 'decision': 'reject score strictly above threshold'}, 'sampler': {'mode': 'class_balanced', 'seed': 42, 'replacement': True, 'draws_per_epoch': 70_576}, 'model': {'hidden_dim': 64, 'num_heads': 4, 'num_layers': 2, 'mlp_dim': 128, 'batch_size': 256, 'learning_rate': 0.001, 'seed': 42, 'mixed_precision': False, 'max_epochs': 12, 'early_stopping_patience': 5}, 'validation': 'full clean seen-class IDs 0-7; minimum loss only', 'test_used_for_selection': False}
protocol_path = output_dir / 'protocol.json'
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == protocol, 'Prior Task 2 output belongs to a different protocol'
else:
    protocol_path.write_text(json.dumps(protocol, indent=2))
env_path = output_dir / 'environment.json'
if env_path.exists():
    assert json.loads(env_path.read_text()) == environment
else:
    env_path.write_text(json.dumps(environment, indent=2))
print('Task 2 output:', output_dir, 'conditions:', conditions)


## 4. Freeze clean-validation calibration, corrupt replay once, and gate only replay

Score clean Task 1 validation rows with the frozen five-output teacher in bounded batches. The gate threshold is the predeclared 95th percentile of `1 - p_teacher(clean supplied label)`; neither Task 2 nor test rows calibrate it. Targeted corruption uses seed 42: among 400 eligible replay attack rows (IDs 1–4), change 80 labels to Benign (20% eligible, 16% of 500 total). The unfiltered and filtered attacked arms share this **exact** corrupted buffer. The gate receives replay features, *supplied* labels, teacher probabilities/class IDs and threshold only. Clean original labels and changed positions are passed later to a separate audit, never to the gate or for label repair. Rejection is an experimental exclusion, not a persistent review/release system.


In [ ]:
calibration_path = output_dir / 'calibration.json'
if calibration_path.exists():
    calibration_record = json.loads(calibration_path.read_text())
    assert calibration_record['task1_best_sha256'] == task1_best_sha and calibration_record['data_identity'] == task1_identity
    assert calibration_record['fit_scope'] == 'clean Task 1 validation only' and calibration_record['quantile'] == 0.95
    threshold = calibration_record['threshold']
    print('Reusing frozen clean-validation calibration:', threshold)
else:
    task1_validation = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='validation', class_ids=OLD_IDS)
    assert len(task1_validation) == task1_identity['task1_validation_rows']
    validation_scores = []
    for start in range(0, len(task1_validation), 256):
        x, labels, _ = task1_validation.batch(task1_validation.indices[start:start + 256])
        probabilities = teacher.predict_proba(x)
        validation_scores.append(label_inconsistency_scores(probabilities, labels, teacher.class_ids))
    clean_scores = np.concatenate(validation_scores)
    calibration = calibrate_label_consistency_scores(clean_scores, quantile=0.95)
    assert calibration.validation_count == len(task1_validation)
    threshold = calibration.threshold
    calibration_record = {'task1_best_sha256': task1_best_sha, 'data_identity': task1_identity, 'fit_scope': 'clean Task 1 validation only', **asdict(calibration)}
    calibration_path.write_text(json.dumps(calibration_record, indent=2))
    print('Calibrated threshold:', threshold, 'rows:', calibration.validation_count)
attack = apply_label_flip(replay_y, 0.20, 42, mode='targeted', source_class_ids=[1, 2, 3, 4], target_class_id=0, allowed_class_ids=list(OLD_IDS), original_row_indices=replay_ids)
assert attack.eligible_count == 400 and attack.changed_count == 80 and attack.total_count == 500
assert attack.achieved_eligible_fraction == 0.20 and attack.achieved_total_fraction == 0.16
assert np.array_equal(attack.changed_original_row_indices, replay_ids[attack.changed_indices])
attack_record = {'mode': attack.mode, 'seed': attack.seed, 'requested_rate': attack.requested_rate, 'budget_denominator': attack.budget_denominator, 'eligible_count': attack.eligible_count, 'total_replay_rows': attack.total_count, 'changed_count': attack.changed_count, 'achieved_eligible_fraction': attack.achieved_eligible_fraction, 'achieved_total_fraction': attack.achieved_total_fraction, 'source_class_ids': list(attack.source_class_ids), 'target_class_id': attack.target_class_id, 'changed_buffer_positions': attack.changed_indices.tolist(), 'changed_original_row_ids': attack.changed_original_row_indices.tolist()}
attack_path = output_dir / 'attack_audit.json'
if attack_path.exists():
    assert json.loads(attack_path.read_text()) == attack_record
else:
    attack_path.write_text(json.dumps(attack_record, indent=2))
attack_buffer_path = output_dir / 'attacked_replay_labels.npz'
if attack_buffer_path.exists():
    with np.load(attack_buffer_path, allow_pickle=False) as saved:
        assert np.array_equal(saved['supplied_labels'], attack.poisoned_labels) and np.array_equal(saved['original_row_ids'], replay_ids)
else:
    np.savez_compressed(attack_buffer_path, supplied_labels=attack.poisoned_labels, original_row_ids=replay_ids)
teacher_probabilities = teacher.predict_proba(replay_x)
gate = apply_label_consistency_gate(replay_x, attack.poisoned_labels, teacher_probabilities, teacher.class_ids, threshold)
assert np.array_equal(gate.retained_labels, attack.poisoned_labels[gate.retained_mask])
assert np.array_equal(gate.retained_features, replay_x[gate.retained_mask])
gate_audit = replay_gate_metrics(gate.retained_mask, attack.changed_indices, replay_y, attack.poisoned_labels, class_ids=OLD_IDS)  # simulator truth enters audit only
gate_record = {'task1_best_sha256': task1_best_sha, 'calibration_threshold': threshold, 'original_row_ids': replay_ids.tolist(), 'inconsistency_scores': gate.scores.tolist(), 'retained_mask': gate.retained_mask.tolist(), 'retained_original_row_ids': replay_ids[gate.retained_mask].tolist(), 'rejected_original_row_ids': replay_ids[~gate.retained_mask].tolist(), 'audit': gate_audit}
gate_path = output_dir / 'gate_audit.json'
if gate_path.exists():
    assert json.loads(gate_path.read_text()) == gate_record
else:
    gate_path.write_text(json.dumps(gate_record, indent=2))
assert len(gate.retained_labels) == gate_audit['retained_total']
print('Attack budget:', attack.changed_count, '/', attack.eligible_count, 'eligible; full-buffer fraction:', attack.achieved_total_fraction)
print('Gate audit:', gate_audit, 'retained per supplied class:', gate_audit['retained_per_supplied_class'])


## 5. Train three independent Task 2 arms with equal draws

All arms load the **same** Task 1 best checkpoint separately, reset the same expansion seed and verify identical expanded initial weight hashes. Each samples from a disk-backed candidate pool containing all 70,076 clean new-class training rows plus its assigned old replay candidates; replacement means an individual epoch need not draw every unique new row. Uniform-over-present-class sampling then uniform-within-class sampling draws with replacement; exactly 70,576 rows and 276 optimizer steps are requested per epoch for every arm, even after filtering. This holds per-epoch work fixed, but early stopping may produce different **total** work. No replay feature is standardized twice, and no whole feature matrix is allocated. Full clean seen-class validation IDs 0–7 selects each best checkpoint by loss. Each arm has separate durable latest/best/history, strict epoch-boundary resume, and actual exposure counts.


In [ ]:
def array_sha256(array):
    value = np.ascontiguousarray(array)
    return hashlib.sha256(value.tobytes()).hexdigest()
def state_hash(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.network.state_dict().items()):
        digest.update(name.encode()); digest.update(tensor.detach().cpu().numpy().tobytes())
    return digest.hexdigest()
task2_view = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='train', class_ids=NEW_IDS)
validation_view = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='validation', class_ids=ALL_IDS)
assert len(task2_view) == 70_076 and len(validation_view) == 1_375_139
arm_inputs = {'clean_replay': (replay_x, replay_y, replay_ids), 'targeted20_unfiltered': (replay_x, attack.poisoned_labels, replay_ids), 'targeted20_filtered': (gate.retained_features, gate.retained_labels, replay_ids[gate.retained_mask])}
assert tuple(arm_inputs) == conditions and np.array_equal(arm_inputs['targeted20_unfiltered'][1], attack.poisoned_labels)
expanded_initial_hash = None
arm_summaries = {}
for name in conditions:
    folder = output_dir / name; folder.mkdir(exist_ok=True)
    X_replay, y_replay, ids_replay = arm_inputs[name]
    train_view = Task2ReplayView(task2_view, X_replay, y_replay, ids_replay, class_ids=ALL_IDS, draws_per_epoch=70_576)
    random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    model = TabularTransformerClassifier.load(best_path, device='cuda')
    assert model.class_ids == OLD_IDS
    torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    model.add_classes(NEW_IDS)
    model.config = replace(model.config, epochs=12, early_stopping_patience=5, training_sampler='class_balanced', sampler_seed=42)
    assert model.class_ids == ALL_IDS and model.config.num_classes == 8
    initial_hash = state_hash(model)
    if expanded_initial_hash is None:
        expanded_initial_hash = initial_hash
    assert initial_hash == expanded_initial_hash, 'Expanded initial weights differ across arms'
    assert (model.config.hidden_dim, model.config.num_heads, model.config.num_layers, model.config.mlp_dim, model.config.batch_size, model.config.learning_rate, model.config.seed, model.config.mixed_precision) == (64, 4, 2, 128, 256, 0.001, 42, False)
    arm_identity = {'source_sha': checked_out_sha, 'task1_best_sha256': task1_best_sha, 'task1_data_identity': task1_identity, 'condition': name, 'task2_training_rows': 70_076, 'validation_rows': len(validation_view), 'draws_per_epoch': 70_576, 'replay_features_sha256': array_sha256(X_replay), 'replay_supplied_labels_sha256': array_sha256(y_replay), 'replay_original_ids_sha256': array_sha256(ids_replay), 'calibration_threshold': threshold, 'expanded_initial_sha256': initial_hash}
    arm_config = {'condition': name, 'source_sha': checked_out_sha, 'model': asdict(model.config), 'data_identity': arm_identity, 'present_supplied_classes': list(train_view.present_class_ids), 'stored_replay_rows': len(y_replay), 'fixed_draws_per_epoch': len(train_view), 'checkpoint_selection': 'minimum full clean seen-class validation loss only'}
    arm_config_path = folder / 'arm_config.json'
    if arm_config_path.exists():
        assert json.loads(arm_config_path.read_text()) == arm_config
    else:
        arm_config_path.write_text(json.dumps(arm_config, indent=2))
    latest_path = folder / 'latest.pt'
    if not latest_path.exists() and any((folder / filename).exists() for filename in ('best.pt', 'history.json', 'training_manifest.json', 'metrics.json')):
        raise RuntimeError(f'{name}: partial arm folder without latest.pt; inspect before restarting')
    result = train_full_disk_backed(model, train_view, validation_view, output_dir=folder, data_identity=arm_identity, resume=latest_path.exists(), max_epochs_this_call=MAX_EPOCHS_THIS_CALL, progress_every_batches=100, report=lambda message: print(message, flush=True))
    if result['paused']:
        raise RuntimeError(f'{name} paused after completed epoch {result["completed_epoch"]}; save entire output folder and set RESUME_RUN_DIR to continue')
    assert result['history']['training_rows_processed'] == 70_576 * result['completed_epoch']
    assert result['history']['optimizer_steps'] == 276 * result['completed_epoch']
    for row in result['history']['epochs']:
        exposure = row['sampling_exposure']
        assert row['train_rows'] == exposure['draws'] == 70_576 and sum(exposure['sampled_counts'].values()) == 70_576
        assert exposure['stored_replay_rows'] == len(y_replay) and exposure['sampler_seed'] == 42
    latest = torch.load(latest_path, map_location='cpu', weights_only=False)
    best_model = TabularTransformerClassifier.load(folder / 'best.pt', device='cuda')
    assert best_model.class_ids == ALL_IDS and latest['best_epoch'] == result['best_epoch']
    assert all(torch.equal(t.detach().cpu(), latest['best_state_dict'][key]) for key, t in best_model.network.state_dict().items())
    x_check, _, _ = validation_view.batch(validation_view.indices[:256])
    reloaded = TabularTransformerClassifier.load(folder / 'best.pt', device='cuda')
    assert np.array_equal(best_model.predict(x_check), reloaded.predict(x_check))
    reload_record = {'best_epoch': result['best_epoch'], 'latest_best_state_equal': True, 'bounded_validation_predictions_identical': True, 'best_checkpoint_sha256': file_sha256(folder / 'best.pt')}
    reload_path = folder / 'reload_verification.json'
    if reload_path.exists():
        assert json.loads(reload_path.read_text()) == reload_record
    else:
        reload_path.write_text(json.dumps(reload_record, indent=2))
    arm_summaries[name] = {'completed_epoch': result['completed_epoch'], 'best_epoch': result['best_epoch'], 'best_validation_loss': result['best_validation_loss'], 'early_stopped': result['early_stopped'], 'training_draws': result['history']['training_rows_processed'], 'validation_rows_processed': result['history']['validation_rows_processed'], 'optimizer_steps': result['history']['optimizer_steps'], 'train_seconds': sum(row['train_seconds'] for row in result['history']['epochs']), 'validation_seconds': sum(row['validation_seconds'] for row in result['history']['epochs']), 'stored_replay_rows': len(y_replay), 'present_supplied_classes': list(train_view.present_class_ids), 'exposure_by_epoch': [row['sampling_exposure'] for row in result['history']['epochs']], 'sampled_counts_total': {str(class_id): sum(row['sampling_exposure']['sampled_counts'][str(class_id)] for row in result['history']['epochs']) for class_id in ALL_IDS}, 'expanded_initial_sha256': initial_hash, 'best_checkpoint_sha256': reload_record['best_checkpoint_sha256'], 'checkpoint_reload_verified': True}
    arm_summary_path = folder / 'arm_summary.json'
    if arm_summary_path.exists():
        assert json.loads(arm_summary_path.read_text()) == arm_summaries[name]
    else:
        arm_summary_path.write_text(json.dumps(arm_summaries[name], indent=2))
    print(name, 'epochs:', result['completed_epoch'], 'draws:', arm_summaries[name]['training_draws'], 'steps:', arm_summaries[name]['optimizer_steps'], 'best:', result['best_epoch'])


## 6. Evaluate frozen best checkpoints, once per arm and partition

Only after validation-loss selection, open old and new test views with frozen Task-1-only preprocessing. Bounded inference accumulates eight-class confusion counts; the combined result is their sum, so test rows are not inferred a third time. The saved five-class Task 1 test confusion is the pre-Task-2 reference. Focused old F1 averages IDs 0–4 with new-class predictions counted as errors; focused new F1 averages 5–7. Forgetting is *before minus after*, so negative values are improvements. Report Benign FPR and old-attack (1–4) to Benign errors. Metrics are observation only, never a tuning criterion. Existing matching metrics are reused after interruption.


In [ ]:
old_test_ids = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='test', class_ids=OLD_IDS).indices
new_test_ids = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='test', class_ids=NEW_IDS).indices
old_test = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='test', class_ids=ALL_IDS, row_ids=old_test_ids)
new_test = DiskBackedFlowDataset(prepared_dir, preprocessor, partition='test', class_ids=ALL_IDS, row_ids=new_test_ids)
assert len(old_test) == 1_360_119 and len(new_test) == 15_015
before_metrics = task1_reference['metrics']
class_names = dict(enumerate(BROAD_CLASS_ORDER))
arm_metrics = {}
for name in conditions:
    folder = output_dir / name
    metrics_path = folder / 'metrics.json'
    best_sha = arm_summaries[name]['best_checkpoint_sha256']
    if metrics_path.exists():
        saved = json.loads(metrics_path.read_text())
        assert saved['best_checkpoint_sha256'] == best_sha and saved['task1_best_sha256'] == task1_best_sha
        assert saved['source_sha'] == checked_out_sha and saved['data_identity'] == task1_identity
        summary = saved['summary']
        print(name, 'reused matching saved test metrics; no repeat inference')
    else:
        selected = TabularTransformerClassifier.load(folder / 'best.pt', device='cuda')
        assert selected.class_ids == ALL_IDS
        torch.cuda.synchronize(); start = perf_counter()
        old_result = evaluate_disk_backed(selected, old_test, batch_size=256, class_names=class_names, benign_class_id=0)
        new_result = evaluate_disk_backed(selected, new_test, batch_size=256, class_names=class_names, benign_class_id=0)
        torch.cuda.synchronize(); inference_seconds = perf_counter() - start
        summary = summarize_task2_counts(before_metrics, np.asarray(old_result['confusion_matrix'], dtype=np.int64), np.asarray(new_result['confusion_matrix'], dtype=np.int64), class_names=class_names)
        assert summary['old_test']['total_rows'] == len(old_test) and summary['new_test']['total_rows'] == len(new_test)
        saved = {'condition': name, 'source_sha': checked_out_sha, 'data_identity': task1_identity, 'task1_best_sha256': task1_best_sha, 'best_checkpoint_sha256': best_sha, 'test_inference_seconds': inference_seconds, 'summary': summary}
        metrics_path.write_text(json.dumps(saved, indent=2))
    arm_metrics[name] = summary
    combined_path = folder / 'combined_test_metrics.json'
    if combined_path.exists():
        assert json.loads(combined_path.read_text()) == summary['combined_test']
    else:
        combined_path.write_text(json.dumps(summary['combined_test'], indent=2))
    plot_names = ('combined_confusion_counts.png', 'combined_confusion_row_normalized.png', 'combined_per_class_recall_f1.png')
    existing = [(folder / filename).exists() for filename in plot_names]
    if any(existing) and not all(existing):
        raise RuntimeError(f'{name}: incomplete plots; inspect before rerun')
    if not any(existing):
        print(name, 'saved confusion plots:', save_result_plots(combined_path, folder, prefix='combined'))
    print(name, 'old/new/combined accuracy:', summary['old_test']['accuracy'], summary['new_test']['accuracy'], summary['combined_test']['accuracy'], 'signed forgetting:', summary['forgetting'])


## 7. Compare only the predeclared conditions and preserve every output

`comparison.json` joins the three already saved conditions, attack denominators, calibration/gate audit, training exposure/work, reload checks, and observed classification/forgetting. The charts are derived from saved metrics only. Filtering may reject poisoned rows without improving downstream accuracy; this run tests that, but is not a review/release system or a new attack family. Save the **entire** `/kaggle/working/full_task2_targeted_...` folder as a Kaggle version or private dataset for inspection and any epoch-boundary continuation. Do not commit checkpoints, prepared arrays, or results.


In [ ]:
before_confusion = np.asarray(before_metrics['confusion_matrix'], dtype=np.int64)
before_attack_count = int(before_confusion[1:5, 0].sum())
before_attack_support = int(before_confusion[1:5].sum())
comparison = {'source_sha': checked_out_sha, 'task1_source_sha': EXPECTED_TASK1_SOURCE_SHA, 'task1_best_sha256': task1_best_sha, 'prepared_input_sha256': EXPECTED_INPUT_SHA256, 'preprocessing_sha256': task1_identity['continual_preprocessing_sha256'], 'conditions': list(conditions), 'same_expanded_initial_sha256': expanded_initial_hash, 'fixed_draws_per_epoch': 70_576, 'task1_before': {'accuracy': before_metrics['accuracy'], 'macro_f1': before_metrics['macro_f1'], 'benign_fpr': before_metrics['benign_false_positive']['rate'], 'old_attack_to_benign': {'count': before_attack_count, 'support': before_attack_support, 'rate': before_attack_count / before_attack_support}}, 'attack_budget': attack_record, 'calibration': calibration_record, 'gate_audit': gate_record, 'arms': {name: {'training': arm_summaries[name], 'metrics': arm_metrics[name]} for name in conditions}, 'test_used_for_selection': False}
comparison_path = output_dir / 'comparison.json'
if comparison_path.exists():
    assert json.loads(comparison_path.read_text()) == comparison
else:
    comparison_path.write_text(json.dumps(comparison, indent=2))
chart_path = output_dir / 'forgetting_per_class_comparison.png'
if not chart_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(23, 5))
    positions = np.arange(len(conditions))
    axes[0].bar(positions - 0.18, [arm_metrics[name]['forgetting']['accuracy_forgetting'] for name in conditions], width=0.36, label='Old accuracy forgetting')
    axes[0].bar(positions + 0.18, [arm_metrics[name]['forgetting']['macro_f1_forgetting'] for name in conditions], width=0.36, label='Old macro-F1 forgetting')
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].set_xticks(positions, conditions, rotation=25, ha='right'); axes[0].set(ylabel='Before minus after', title='Signed Task 1 forgetting'); axes[0].legend()
    x = np.arange(8); width = 0.26
    for i, name in enumerate(conditions):
        per_class = arm_metrics[name]['combined_test']['per_class']
        axes[1].bar(x + (i - 1) * width, [per_class[str(class_id)]['recall'] for class_id in ALL_IDS], width=width, label=name)
    axes[1].set_xticks(x, BROAD_CLASS_ORDER, rotation=45, ha='right'); axes[1].set(ylabel='Recall', ylim=(0, 1.05), title='Combined-test per-class recall'); axes[1].legend(fontsize=7)
    for i, name in enumerate(conditions):
        per_class = arm_metrics[name]['combined_test']['per_class']
        axes[2].bar(x + (i - 1) * width, [per_class[str(class_id)]['f1'] for class_id in ALL_IDS], width=width, label=name)
    axes[2].set_xticks(x, BROAD_CLASS_ORDER, rotation=45, ha='right'); axes[2].set(ylabel='F1', ylim=(0, 1.05), title='Combined-test per-class F1'); axes[2].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(chart_path, dpi=160); plt.show()
print('All three condition folders complete. Save:', output_dir)
print('Return executed notebook, protocol.json, environment.json, calibration.json, attack_audit.json, gate_audit.json, comparison.json, forgetting_per_class_comparison.png, and each condition folders arm_config.json, arm_summary.json, history.json, training_manifest.json, reload_verification.json, metrics.json, combined_test_metrics.json and plots. Preserve all best.pt/latest.pt and attacked_replay_labels.npz privately.')
